<a href="https://colab.research.google.com/github/rodrigoesborges/conju-python/blob/main/Grupo_6_desemprego.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-

# %% [célula 1] Instalação (Instalação automática das dependências no Colab)
!pip install -q wbgapi pandas requests

# %% [célula 2] Imports e configuração
from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import requests

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("desemprego")

# Mapeamento de países entre as APIs (ISO3 é usado por BM, FMI e CEPALSTAT)
PAISES: Dict[str, Dict[str, str]] = {
    "Brasil":    {"iso3": "BRA", "cepal_nomes": ["brazil", "brasil"]},
    "México":    {"iso3": "MEX", "cepal_nomes": ["mexico", "méxico"]},
    "Argentina": {"iso3": "ARG", "cepal_nomes": ["argentina"]},
    "Colômbia":  {"iso3": "COL", "cepal_nomes": ["colombia", "colômbia"]},
    "Chile":     {"iso3": "CHL", "cepal_nomes": ["chile"]},
}
ISO3_PARA_PAIS = {v["iso3"]: k for k, v in PAISES.items()}

JANELA = {"mensal": 12, "trimestral": 4, "anual": 1}


@dataclass
class Indicador:
    """Metadados de uma série configurada."""
    nome: str                 # nome exibido na tabela final
    fonte: str                # 'bm' | 'fmi' | 'cepal'
    codigo: str               # código da série na API
    frequencia: str           # 'mensal' | 'trimestral' | 'anual'
    tipo_variavel: str        # 'fluxo' | 'estoque' | 'taxa'
    metrica_final: str        # 'nivel' (T1/T2) | 'variacao' (T3/T4)
    agregacao_estoque: str = "media"   # 'media' | 'ultimo'
    # Para taxas de nível (desemprego %), variação é em p.p. (diferença).
    # Use 'razao' para (v/v_ant - 1).
    tipo_variacao: str = "diferenca"


# %% [célula 3] Etapa 0 - Extração
def extrair_bm(ind: Indicador, paises: List[str], ano_ini: int, ano_fim: int) -> pd.DataFrame:
    """Extrai série anual do Banco Mundial via wbgapi."""
    import wbgapi as wb
    isos = [PAISES[p]["iso3"] for p in paises]
    df = wb.data.DataFrame(ind.codigo, isos, time=range(ano_ini, ano_fim + 1),
                           labels=False, columns="series").reset_index()
    df.columns = ["iso3", "ano", "valor"]
    df["ano"] = df["ano"].astype(str).str.replace("YR", "").astype(int)
    df["periodo"] = pd.to_datetime(df["ano"].astype(str) + "-01-01")
    return df[["iso3", "periodo", "valor"]]


def extrair_fmi(ind: Indicador, paises: List[str], ano_ini: int, ano_fim: int) -> pd.DataFrame:
    """Extrai série anual do FMI via API REST do DataMapper (ex.: LUR)."""
    isos = "/".join(PAISES[p]["iso3"] for p in paises)
    url = f"https://www.imf.org/external/datamapper/api/v1/{ind.codigo}/{isos}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    valores = r.json().get("values", {}).get(ind.codigo, {})
    linhas = []
    for iso, serie in valores.items():
        for ano, v in serie.items():
            if ano_ini <= int(ano) <= ano_fim:
                linhas.append((iso, pd.to_datetime(f"{ano}-01-01"), v))
    return pd.DataFrame(linhas, columns=["iso3", "periodo", "valor"])


def _periodo_cepal(texto: str) -> Optional[pd.Timestamp]:
    """Converte rótulos de tempo da CEPALSTAT (2020, 2020-03, 2020 Q1 ...)."""
    t = str(texto).strip()
    try:
        if len(t) == 4 and t.isdigit():
            return pd.Timestamp(f"{t}-01-01")
        if "Q" in t.upper():
            ano, q = t.upper().replace("-", " ").split("Q")[0].strip(), t.upper().split("Q")[-1].strip()
            return pd.Timestamp(year=int(ano), month=3 * int(q) - 2, day=1)
        return pd.to_datetime(t).replace(day=1)
    except Exception:
        return None


def extrair_cepal(ind: Indicador, paises: List[str], ano_ini: int, ano_fim: int) -> pd.DataFrame:
    """
    Extrai da API REST da CEPALSTAT (melhor esforço: o esquema de dimensões varia
    por indicador). Retorna colunas iso3, periodo, valor.
    """
    url = f"https://api-cepalstat.cepal.org/cepalstat/api/v1/indicator/{ind.codigo}/data"
    r = requests.get(url, params={"lang": "en", "format": "json"}, timeout=90)
    r.raise_for_status()
    corpo = r.json().get("body", {})
    dims = {d["id"]: {m["id"]: m for m in d["members"]} for d in corpo.get("dimensions", [])}

    nomes_para_iso = {n: v["iso3"] for v in PAISES.values() for n in v["cepal_nomes"]}
    linhas = []
    for reg in corpo.get("data", []):
        iso, per = None, None
        for chave, mid in reg.items():
            if not chave.startswith("dim_"):
                continue
            membro = dims.get(int(chave[4:]), {}).get(mid)
            if not membro:
                continue
            nome = str(membro.get("name", "")).lower()
            if nome in nomes_para_iso:
                iso = nomes_para_iso[nome]
            elif _periodo_cepal(membro.get("name")) is not None:
                per = _periodo_cepal(membro.get("name"))
        if iso and per is not None and ano_ini <= per.year <= ano_fim:
            try:
                linhas.append((iso, per, float(reg["value"])))
            except (KeyError, TypeError, ValueError):
                continue
    df = pd.DataFrame(linhas, columns=["iso3", "periodo", "valor"])
    if df.empty:
        log.warning("CEPALSTAT sem dados p/ %s - confira o ID do indicador e as dimensões "
                    "(ex.: filtre sexo/idade, se houver).", ind.codigo)
    # Se sobrar mais de uma linha por país/período (outras dimensões), tira a média
    return df.groupby(["iso3", "periodo"], as_index=False)["valor"].mean()


EXTRATORES = {"bm": extrair_bm, "fmi": extrair_fmi, "cepal": extrair_cepal}


def extrair_serie(ind: Indicador, paises: List[str], data_inicio: str, data_fim: str) -> pd.DataFrame:
    """Função unificada: devolve dados brutos (iso3, periodo, valor) de uma série."""
    ini, fim = pd.Timestamp(data_inicio), pd.Timestamp(data_fim)
    try:
        df = EXTRATORES[ind.fonte](ind, paises, ini.year, fim.year)
    except Exception as e:  # falha de requisição, parse, etc.
        log.error("Falha ao extrair '%s' (%s): %s", ind.nome, ind.fonte, e)
        return pd.DataFrame(columns=["iso3", "periodo", "valor"])
    df = df[(df["periodo"] >= ini) & (df["periodo"] <= fim)].copy()
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    return df


# %% [célula 4] Etapa 1 - Tabela base (formato longo)
def montar_base(inds: List[Indicador], paises: List[str],
                data_inicio: str, data_fim: str) -> pd.DataFrame:
    """Consolida todas as séries em df_base: pais | periodo | indicador | valor."""
    partes = []
    for ind in inds:
        df = extrair_serie(ind, paises, data_inicio, data_fim)
        if df.empty:
            continue
        df["pais"] = df["iso3"].map(ISO3_PARA_PAIS)
        df["indicador"] = ind.nome
        partes.append(df[["pais", "periodo", "indicador", "valor"]])
    if not partes:
        raise RuntimeError("Nenhuma série foi extraída. Verifique conexão e códigos.")
    base = pd.concat(partes, ignore_index=True)
    return base.dropna(subset=["pais"]).sort_values(["pais", "indicador", "periodo"]).reset_index(drop=True)


# %% [célula 5] Etapa 2 - Acumulações
def _agregar(s: pd.Series, tipo: str, agregacao_estoque: str = "media") -> float:
    """Aplica a regra da natureza da variável a uma janela de valores."""
    if s.isna().any():
        return np.nan
    if tipo == "fluxo":
        return s.sum()
    if tipo == "taxa":
        return float(np.prod(1 + s / 100.0) - 1) * 100.0   # taxas em %
    return s.iloc[-1] if agregacao_estoque == "ultimo" else s.mean()


def acumulado_12m(s: pd.Series, freq: str, tipo: str, agregacao_estoque: str = "media") -> pd.Series:
    """Janela móvel (12 meses / 4 trimestres / 1 ano). Janela incompleta -> NaN."""
    n = JANELA[freq]
    return s.rolling(n, min_periods=n).apply(
        lambda w: _agregar(pd.Series(w), tipo, agregacao_estoque), raw=False)


def acumulado_ano(df: pd.DataFrame, freq: str, tipo: str, agregacao_estoque: str = "media") -> pd.Series:
    """
    YTD dentro do ano-calendário. Se o ano estiver sem os primeiros períodos
    (início da série incompleto), o resultado é NaN.
    """
    n = JANELA[freq]
    df = df.sort_values("periodo")
    out = pd.Series(np.nan, index=df.index)
    for _, g in df.groupby(df["periodo"].dt.year):
        for i in range(len(g)):
            esperado = i + 1
            ate = g.iloc[: i + 1]
            pos_no_ano = (ate["periodo"].iloc[-1].month - 1) // (12 // n) + 1
            if esperado != pos_no_ano:    # início do ano faltando
                continue
            out.loc[ate.index[-1]] = _agregar(ate["valor"], tipo, agregacao_estoque)
    return out


# %% [célula 6] Etapa 3 - Tabelas intermediárias
def variacao_yoy(t: pd.DataFrame, tipo_variacao: str = "diferenca") -> pd.DataFrame:
    """Self-join com defasagem de 1 ano; devolve variação YoY (diferença p.p. ou razão)."""
    t = t.copy()
    t["periodo_anterior"] = t["periodo"] - pd.DateOffset(years=1)
    ant = t[["pais", "indicador", "periodo", "valor"]].rename(
        columns={"periodo": "periodo_anterior", "valor": "valor_anterior"})
    m = t.merge(ant, on=["pais", "indicador", "periodo_anterior"], how="left")
    if tipo_variacao == "razao":
        den = m["valor_anterior"].replace(0, np.nan)      # evita divisão por zero
        m["valor"] = m["valor"] / den - 1
    else:
        m["valor"] = m["valor"] - m["valor_anterior"]
    return m[["pais", "indicador", "periodo", "valor"]]


def gerar_intermediarias(base: pd.DataFrame, inds: List[Indicador]):
    """Retorna T1 (YTD), T2 (12m), T3 (YoY de T1), T4 (YoY de T2)."""
    meta = {i.nome: i for i in inds}
    t1, t2 = [], []
    for (pais, nome), g in base.groupby(["pais", "indicador"]):
        m = meta[nome]
        g = g.sort_values("periodo").reset_index(drop=True)
        a = g[["pais", "indicador", "periodo"]].copy()
        b = a.copy()
        a["valor"] = acumulado_ano(g, m.frequencia, m.tipo_variavel, m.agregacao_estoque).values
        b["valor"] = acumulado_12m(g["valor"], m.frequencia, m.tipo_variavel, m.agregacao_estoque).values
        t1.append(a); t2.append(b)
    T1, T2 = pd.concat(t1, ignore_index=True), pd.concat(t2, ignore_index=True)

    def _yoy(T):
        partes = []
        for nome, g in T.groupby("indicador"):
            partes.append(variacao_yoy(g, meta[nome].tipo_variacao))
        return pd.concat(partes, ignore_index=True)

    return T1, T2, _yoy(T1), _yoy(T2)


# %% [célula 7] Etapa 4 - Consolidação final
def tabela_final(base, T1, T2, T3, T4, inds: List[Indicador]) -> pd.DataFrame:
    """Últimas 3 observações (wide) + acumulados do último período disponível."""
    meta = {i.nome: i for i in inds}
    blocos = []
    for nome, m in meta.items():
        b = base[base["indicador"] == nome]
        if b.empty:
            continue
        ult3 = (b.sort_values("periodo").groupby("pais").tail(3)
                .pivot(index=["pais", "indicador"], columns="periodo", values="valor"))
        ult3.columns = [c.strftime("%Y-%m-%d") for c in ult3.columns]
        ultimo = b["periodo"].max()
        ta, tb = (T1, T2) if m.metrica_final == "nivel" else (T3, T4)
        rot = "" if m.metrica_final == "nivel" else " (var. YoY)"
        ref = ultimo.strftime("%Y-%m")
        def pega(T, col):
            x = T[(T["indicador"] == nome) & (T["periodo"] == ultimo)]
            return x.set_index(["pais", "indicador"])["valor"].rename(col)
        blocos.append(ult3.join(pega(ta, f"Ac. no Ano{rot} (até {ref})"))
                          .join(pega(tb, f"Ac. 12 Meses{rot} (até {ref})")))
    return pd.concat(blocos).reset_index()


# %% [célula 8] Exemplo executável (somente desemprego)
if __name__ == "__main__" or True:
    INDICADORES = [
        Indicador("Desemprego BM (% força de trabalho)", "bm", "SL.UEM.TOTL.ZS",
                  "anual", "estoque", "nivel"),
        Indicador("Desemprego FMI (% força de trabalho)", "fmi", "LUR",
                  "anual", "estoque", "variacao"),
        # CEPALSTAT: troque o código pelo ID do indicador de desemprego desejado
        # (consulte em statistics.cepal.org; exemplo de formato abaixo - CONFIRA).
        # Indicador("Desemprego CEPAL", "cepal", "<ID_DO_INDICADOR>",
        #           "anual", "estoque", "nivel"),
    ]

    DATA_INICIO, DATA_FIM = "2015-01-01", "2025-12-31"   # FMI inclui projeções do WEO
    df_base = montar_base(INDICADORES, list(PAISES), DATA_INICIO, DATA_FIM)
    T1, T2, T3, T4 = gerar_intermediarias(df_base, INDICADORES)
    df_final = tabela_final(df_base, T1, T2, T3, T4, INDICADORES)

    pd.set_option("display.width", 200, "display.max_columns", 20)
    print(df_final.round(2).to_string(index=False))
    # df_final.to_csv("desemprego_final.csv", index=False)   # no Colab: files.download(...)


     pais                            indicador  2023-01-01  2024-01-01  2025-01-01  Ac. no Ano (até 2025-01)  Ac. 12 Meses (até 2025-01)  Ac. no Ano (var. YoY) (até 2025-01)  Ac. 12 Meses (var. YoY) (até 2025-01)
Argentina  Desemprego BM (% força de trabalho)        6.14        7.15        7.14                      7.14                        7.14                                  NaN                                    NaN
   Brasil  Desemprego BM (% força de trabalho)        7.95        6.80        5.97                      5.97                        5.97                                  NaN                                    NaN
    Chile  Desemprego BM (% força de trabalho)        9.01        8.72        8.97                      8.97                        8.97                                  NaN                                    NaN
 Colômbia  Desemprego BM (% força de trabalho)        9.59        9.62        8.29                      8.29                        8.29            